Import thư viện

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

from imblearn.over_sampling import SMOTE

Hàm đánh giá

In [2]:
results = []

def evaluate_model(y_true, y_pred, y_prob, model_name):

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print("\n[1] Classification Report")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Normal", "Fraud"]
    ))

    cm = confusion_matrix(y_true, y_pred)

    print("\n[2] Confusion Matrix")
    print(f"True Negative  : {cm[0][0]}")
    print(f"False Positive : {cm[0][1]}")
    print(f"False Negative : {cm[1][0]}")
    print(f"True Positive  : {cm[1][1]}")

    print("\n[3] Metrics")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-score  : {f1:.4f}")

    results.append({
        "Model": model_name,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
    })

Load dataset

In [3]:
df = pd.read_csv("creditcard.csv")

print(f"Dataset size = {len(df)}")
print(f"Fraud = {df['Class'].sum()}")
print(f"Fraud ratio = {df['Class'].mean()*100:.4f}%")

feature_cols = [f"V{i}" for i in range(1, 29)] + ["Amount", "Time"]

X = df[feature_cols].copy()
y = df["Class"]

Dataset size = 284807
Fraud = 492
Fraud ratio = 0.1727%


Scale

In [4]:
scaler_amount = StandardScaler()
scaler_time = StandardScaler()

X["Amount"] = scaler_amount.fit_transform(X[["Amount"]])
X["Time"] = scaler_time.fit_transform(X[["Time"]])

Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

SMOTE

In [6]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("Normal =", sum(y_train_smote == 0))
print("Fraud =", sum(y_train_smote == 1))

Normal = 227451
Fraud = 227451


Tạo thư mục chứa model

In [7]:
import os

os.makedirs("model", exist_ok=True)

Logistic Regression

In [8]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_smote, y_train_smote)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print('=== LOGISTIC REGRESSION ===')

evaluate_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression")

joblib.dump(
    {
        "model": lr,
        "scaler_amount": scaler_amount,
        "scaler_time": scaler_time
    },
    "model/lr.pkl"
)

print("Đã lưu model Logistic Regression")

=== LOGISTIC REGRESSION ===

[1] Classification Report
              precision    recall  f1-score   support

      Normal       1.00      0.97      0.99     56864
       Fraud       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962


[2] Confusion Matrix
True Negative  : 55406
False Positive : 1458
False Negative : 8
True Positive  : 90

[3] Metrics
Precision : 0.0581
Recall    : 0.9184
F1-score  : 0.1094
Đã lưu model Logistic Regression


Random Forest

In [9]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_smote, y_train_smote) 
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== RANDOM FOREST ===')
evaluate_model(y_test, y_pred_rf, y_prob_rf, "Random Forest")

joblib.dump(
    {
        "model": rf,
        "scaler_amount": scaler_amount,
        "scaler_time": scaler_time
    },
    "model/rf.pkl"
)

print("Đã lưu model Random Forest")

=== RANDOM FOREST ===

[1] Classification Report
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.82      0.82      0.82        98

    accuracy                           1.00     56962
   macro avg       0.91      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962


[2] Confusion Matrix
True Negative  : 56847
False Positive : 17
False Negative : 18
True Positive  : 80

[3] Metrics
Precision : 0.8247
Recall    : 0.8163
F1-score  : 0.8205
Đã lưu model Random Forest


Isolation Forest

In [10]:
iso = IsolationForest(contamination=0.0017, random_state=42)
iso.fit(X_train)
y_raw = iso.predict(X_test)
y_pred_iso = [1 if v == -1 else 0 for v in y_raw]
y_prob_iso = [1.0 if v == -1 else 0.0 for v in y_raw]

print('=== ISOLATION FOREST ===')
evaluate_model(y_test, y_pred_iso, y_prob_iso, "Isolation Forest")

joblib.dump(
    {
        "model": iso, 
        "scaler_amount": scaler_amount,
        "scaler_time": scaler_time,
        "model_type": "IsolationForest"
    },
    "model/if.pkl"
)

print("Đã lưu model Isolation Forest")

=== ISOLATION FOREST ===

[1] Classification Report
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.31      0.33      0.32        98

    accuracy                           1.00     56962
   macro avg       0.65      0.66      0.66     56962
weighted avg       1.00      1.00      1.00     56962


[2] Confusion Matrix
True Negative  : 56792
False Positive : 72
False Negative : 66
True Positive  : 32

[3] Metrics
Precision : 0.3077
Recall    : 0.3265
F1-score  : 0.3168
Đã lưu model Isolation Forest


Tổng hợp kết quả

In [11]:
result_df = pd.DataFrame(results)

result_df = result_df.sort_values(
    by="F1-score",
    ascending=False
)

print(result_df.round(4))

                 Model  Precision  Recall  F1-score
1        Random Forest     0.8247  0.8163    0.8205
2     Isolation Forest     0.3077  0.3265    0.3168
0  Logistic Regression     0.0581  0.9184    0.1094


Tối ưu tham số 

In [12]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_smote, y_train_smote)

print('Best params:', grid_search.best_params_)
print('Best F1    :', grid_search.best_score_)

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_prob_best = best_rf.predict_proba(X_test)[:, 1]

print('\n=== BEST RANDOM FOREST ===')
evaluate_model(y_test, y_pred_best, y_prob_best, "Optimized Random Forest")

Fitting 3 folds for each of 27 candidates, totalling 81 fits


KeyboardInterrupt: 

Đặc trưng quan trọng 

In [ ]:
importance = rf.feature_importances_
feat_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': importance
}).sort_values('importance', ascending=False)

# In top 10
print(feat_df.head(10).to_string(index=False))

# Biểu đồ
plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df.head(10), x='importance', y='feature',
            palette='viridis')
plt.title('Top 10 Feature Importance – Random Forest')
plt.xlabel('Mức độ quan trọng')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()